# Behavioural Feature Engineering

Behavioural features capture historical customer activity instead of analysing transactions independently.

Unlike static transaction attributes, behavioural features describe spending patterns, transaction frequency and deviations from historical behaviour.

These features are commonly used in production fraud detection systems.

In [1]:
# ============================================================
# Imports
# ============================================================

import pandas as pd
import numpy as np

from pathlib import Path

print("Libraries Loaded")

Libraries Loaded


In [2]:
DATA = Path("../data")

raw = pd.read_csv("financial_fraud_detection_dataset.csv")

raw.head()

,transaction_id,timestamp,sender_account,receiver_account,amount,transaction_type,merchant_category,location,device_used,is_fraud,fraud_type,time_since_last_transaction,spending_deviation_score,velocity_score,geo_anomaly_score,payment_channel,ip_address,device_hash
0,T100000,2023-08-22T09:22:43.516168,ACC877572,ACC388389,343.78,withdrawal,utilities,Tokyo,mobile,False,NaN,NaN,-0.21,3,0.22,card,13.101.214.112,D8536477
1,T100001,2023-08-04T01:58:02.606711,ACC895667,ACC944962,419.65,withdrawal,online,Toronto,atm,False,NaN,NaN,-0.14,7,0.96,ACH,172.52.47.194,D2622631
2,T100002,2023-05-12T11:39:33.742963,ACC733052,ACC377370,2773.86,deposit,other,London,pos,False,NaN,NaN,-1.78,20,0.89,card,185.98.35.23,D4823498
3,T100003,2023-10-10T06:04:43.195112,ACC996865,ACC344098,1666.22,deposit,online,Sydney,pos,False,NaN,NaN,-0.60,6,0.37,wire_transfer,107.136.36.87,D9961380
4,T100004,2023-09-24T08:09:02.700162,ACC584714,ACC497887,24.43,transfer,utilities,Toronto,mobile,False,NaN,NaN,0.79,13,0.27,ACH,108.161.108.255,D7637601


In [3]:
# ============================================================
# Convert Timestamp
# ============================================================

raw["timestamp"] = pd.to_datetime(
    raw["timestamp"],
    format="mixed"
)

raw = raw.sort_values("timestamp")

raw.reset_index(
    drop=True,
    inplace=True
)

print(raw.shape)

(5000000, 18)


In [4]:
raw["historical_avg_amount"] = (

    raw
    .groupby("sender_account")["amount"]
    .expanding()
    .mean()
    .shift()
    .reset_index(level=0, drop=True)

)

In [5]:
raw["historical_max_amount"] = (

    raw
    .groupby("sender_account")["amount"]
    .cummax()
    .shift()

)

In [6]:
raw["historical_txn_count"] = (

    raw
    .groupby("sender_account")
    .cumcount()

)

In [7]:
raw["amount_ratio"] = (

    raw["amount"]

    /

    raw["historical_avg_amount"]

)

# Advanced Feature Engineering

## Objective

The objective of this notebook is to improve fraud detection performance by engineering historical and behavioural features from the original transaction dataset.

Unlike Notebook 2, which focused on preprocessing, this notebook creates new predictive variables that capture customer behaviour over time.

These engineered features will be used in Notebook 5 to retrain machine learning models.

In [9]:
# ============================================================
# Imports
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

pd.set_option("display.max_columns", None)

SEED = 42

print("Libraries Loaded Successfully")

Libraries Loaded Successfully


In [12]:
# ============================================================
# Load Dataset
# ============================================================

DATA = Path("../data")

raw = pd.read_csv(
    "financial_fraud_detection_dataset.csv"
)

print(raw.shape)

raw.head()

(5000000, 18)


,transaction_id,timestamp,sender_account,receiver_account,amount,transaction_type,merchant_category,location,device_used,is_fraud,fraud_type,time_since_last_transaction,spending_deviation_score,velocity_score,geo_anomaly_score,payment_channel,ip_address,device_hash
0,T100000,2023-08-22T09:22:43.516168,ACC877572,ACC388389,343.78,withdrawal,utilities,Tokyo,mobile,False,NaN,NaN,-0.21,3,0.22,card,13.101.214.112,D8536477
1,T100001,2023-08-04T01:58:02.606711,ACC895667,ACC944962,419.65,withdrawal,online,Toronto,atm,False,NaN,NaN,-0.14,7,0.96,ACH,172.52.47.194,D2622631
2,T100002,2023-05-12T11:39:33.742963,ACC733052,ACC377370,2773.86,deposit,other,London,pos,False,NaN,NaN,-1.78,20,0.89,card,185.98.35.23,D4823498
3,T100003,2023-10-10T06:04:43.195112,ACC996865,ACC344098,1666.22,deposit,online,Sydney,pos,False,NaN,NaN,-0.60,6,0.37,wire_transfer,107.136.36.87,D9961380
4,T100004,2023-09-24T08:09:02.700162,ACC584714,ACC497887,24.43,transfer,utilities,Toronto,mobile,False,NaN,NaN,0.79,13,0.27,ACH,108.161.108.255,D7637601


In [13]:
# ============================================================
# Timestamp Processing
# ============================================================

raw["timestamp"] = pd.to_datetime(
    raw["timestamp"],
    format="mixed"
)

raw = raw.sort_values("timestamp")

raw.reset_index(
    drop=True,
    inplace=True)

print("Dataset sorted by timestamp.")

Dataset sorted by timestamp.


In [14]:
# ============================================================
# Historical Transaction Count
# ============================================================

raw["historical_txn_count"] = (

    raw
    .groupby("sender_account")
    .cumcount()

)

In [23]:
# ============================================================
# Historical Average Amount (Leakage-Free)
# ============================================================

raw["historical_avg_amount"] = (
    raw.groupby("sender_account")["amount"]
       .transform(lambda x: x.shift().expanding().mean())
)

In [24]:
# ============================================================
# Historical Maximum Amount (Leakage-Free)
# ============================================================

raw["historical_max_amount"] = (
    raw.groupby("sender_account")["amount"]
       .transform(lambda x: x.shift().cummax())
)

In [25]:
# ============================================================
# Historical Minimum Amount (Leakage-Free)
# ============================================================

raw["historical_min_amount"] = (
    raw.groupby("sender_account")["amount"]
       .transform(lambda x: x.shift().cummin())
)

In [26]:
# ============================================================
# Historical Standard Deviation (Leakage-Free)
# ============================================================

raw["historical_std_amount"] = (
    raw.groupby("sender_account")["amount"]
       .transform(lambda x: x.shift().expanding().std())
)

In [27]:
raw["amount_ratio"] = (
    raw["amount"] /
    raw["historical_avg_amount"].replace(0, np.nan)
)

In [28]:
raw["amount_difference"] = (
    raw["amount"] -
    raw["historical_avg_amount"]
)

In [21]:
raw["previous_timestamp"] = (

    raw
    .groupby("sender_account")["timestamp"]
    .shift()
)

raw["time_gap_minutes"] = (

    raw["timestamp"]

    -

    raw["previous_timestamp"]

).dt.total_seconds() / 60

In [29]:
raw[
    [
        "sender_account",
        "amount",
        "historical_txn_count",
        "historical_avg_amount",
        "historical_max_amount",
        "historical_min_amount",
        "historical_std_amount"
    ]
].head(20)

,sender_account,amount,historical_txn_count,historical_avg_amount,historical_max_amount,historical_min_amount,historical_std_amount
0,ACC259476,12.25,0,NaN,NaN,NaN,NaN
1,ACC478421,1347.27,0,NaN,NaN,NaN,NaN
2,ACC830612,20.79,0,NaN,NaN,NaN,NaN
3,ACC647991,740.73,0,NaN,NaN,NaN,NaN
4,ACC984405,228.67,0,NaN,NaN,NaN,NaN
5,ACC995556,1647.18,0,NaN,NaN,NaN,NaN
6,ACC618264,10.26,0,NaN,NaN,NaN,NaN
7,ACC654646,1626.52,0,NaN,NaN,NaN,NaN
8,ACC287060,2107.20,0,NaN,NaN,NaN,NaN
9,ACC555732,6.65,0,NaN,NaN,NaN,NaN


In [30]:
# Find a sender with more than one transaction
sender = (
    raw["sender_account"]
    .value_counts()
    .loc[lambda x: x > 1]
    .index[0]
)

print("Sample Sender:", sender)

raw.loc[
    raw["sender_account"] == sender,
    [
        "timestamp",
        "sender_account",
        "amount",
        "historical_txn_count",
        "historical_avg_amount",
        "historical_max_amount",
        "historical_min_amount",
        "historical_std_amount"
    ]
].head(10)

Sample Sender: ACC983922


,timestamp,sender_account,amount,historical_txn_count,historical_avg_amount,historical_max_amount,historical_min_amount,historical_std_amount
352795,2023-01-27 05:01:25.332723,ACC983922,469.47,0,NaN,NaN,NaN,NaN
607111,2023-02-14 18:51:53.188771,ACC983922,11.67,1,469.470000,469.47,469.47,NaN
729493,2023-02-23 18:20:04.077658,ACC983922,309.10,2,240.570000,469.47,11.67,323.713484
836018,2023-03-03 13:38:13.986989,ACC983922,8.28,3,263.413333,469.47,11.67,232.294347
935340,2023-03-10 18:57:36.127245,ACC983922,492.82,4,199.630000,469.47,8.28,228.576093
1043696,2023-03-18 17:20:43.523695,ACC983922,13.67,5,258.268000,492.82,8.28,237.439146
1292659,2023-04-05 20:13:33.526095,ACC983922,0.01,6,217.501667,492.82,8.28,234.676888
1330261,2023-04-08 14:27:37.733411,ACC983922,9.78,7,186.431429,492.82,0.01,229.459988
1355484,2023-04-10 10:23:56.763512,ACC983922,424.96,8,164.350000,492.82,0.01,221.429230
1587078,2023-04-27 07:49:09.262579,ACC983922,167.02,9,193.306667,492.82,0.01,224.607297


In [31]:
# ============================================================
# Previous Transaction Amount
# ============================================================

raw["previous_amount"] = (
    raw.groupby("sender_account")["amount"]
       .shift(1)
)

In [32]:
# ============================================================
# Amount Change
# ============================================================

raw["amount_change"] = (
    raw["amount"] -
    raw["previous_amount"]
)

In [33]:
# ============================================================
# Percentage Change
# ============================================================

raw["amount_pct_change"] = (
    raw.groupby("sender_account")["amount"]
       .pct_change()
)

In [34]:
# ============================================================
# Historical Total Amount
# ============================================================

raw["historical_total_amount"] = (
    raw.groupby("sender_account")["amount"]
       .transform(lambda x: x.shift().cumsum())
)

In [35]:
# ============================================================
# Historical Median
# ============================================================

raw["historical_median_amount"] = (
    raw.groupby("sender_account")["amount"]
       .transform(lambda x: x.shift().expanding().median())
)

In [36]:
# ============================================================
# Amount / Historical Median
# ============================================================

raw["amount_median_ratio"] = (
    raw["amount"] /
    raw["historical_median_amount"].replace(0, np.nan)
)

In [37]:
# ============================================================
# Previous Transaction Time
# ============================================================

raw["previous_timestamp"] = (
    raw.groupby("sender_account")["timestamp"]
       .shift()
)

raw["time_gap_minutes"] = (
    raw["timestamp"] -
    raw["previous_timestamp"]
).dt.total_seconds() / 60

In [38]:
raw["time_gap_hours"] = (
    raw["time_gap_minutes"] / 60
)

In [39]:
raw["rapid_transaction"] = (
    raw["time_gap_minutes"] < 5
).astype(int)

In [41]:
# ============================================================
# Days Since First Transaction
# ============================================================

first_timestamp = (
    raw.groupby("sender_account")["timestamp"]
       .transform("min")
)

raw["days_since_first_txn"] = (
    raw["timestamp"] - first_timestamp
).dt.total_seconds() / (24 * 60 * 60)

In [42]:
# ============================================================
# Average Transactions Per Day
# ============================================================

raw["avg_txn_per_day"] = (
    raw["historical_txn_count"] /
    raw["days_since_first_txn"].replace(0, np.nan)
)

In [44]:
# ============================================================
# Sender Lifetime
# ============================================================

first_txn = (
    raw.groupby("sender_account")["timestamp"]
       .transform("min")
)

raw["sender_account_age_days"] = (
    raw["timestamp"] - first_txn
).dt.total_seconds() / (24 * 3600)

In [45]:
# ============================================================
# Sender Activity Rate
# ============================================================

raw["avg_txn_per_day"] = (
    raw["historical_txn_count"] /
    raw["sender_account_age_days"].replace(0, np.nan)
)

In [46]:
# ============================================================
# Unique Receivers
# ============================================================

raw["sender_unique_receivers"] = (
    raw.groupby("sender_account")["receiver_account"]
       .transform("nunique")
)

In [47]:
# ============================================================
# Receiver Diversity
# ============================================================

raw["receiver_diversity_ratio"] = (
    raw["sender_unique_receivers"] /
    (raw["historical_txn_count"] + 1)
)

In [48]:
# ============================================================
# Receiver Transaction Count
# ============================================================

raw["receiver_transaction_count"] = (
    raw.groupby("receiver_account")
       .cumcount()
)

In [49]:
# ============================================================
# Unique Senders
# ============================================================

raw["receiver_unique_senders"] = (
    raw.groupby("receiver_account")["sender_account"]
       .transform("nunique")
)

In [50]:
# ============================================================
# Device Usage Count
# ============================================================

raw["device_usage_count"] = (
    raw.groupby("device_used")
       .cumcount()
)

In [51]:
# ============================================================
# Accounts Per Device
# ============================================================

raw["accounts_per_device"] = (
    raw.groupby("device_used")["sender_account"]
       .transform("nunique")
)

In [52]:
# ============================================================
# Location Transaction Count
# ============================================================

raw["location_transaction_count"] = (
    raw.groupby("location")
       .cumcount()
)

In [53]:
# ============================================================
# Users Per Location
# ============================================================

raw["location_unique_users"] = (
    raw.groupby("location")["sender_account"]
       .transform("nunique")
)

In [54]:
# ============================================================
# Interaction Features
# ============================================================

raw["amount_velocity"] = (
    raw["amount"] *
    raw["velocity_score"]
)

raw["amount_geo"] = (
    raw["amount"] *
    raw["geo_anomaly_score"]
)

raw["amount_spending"] = (
    raw["amount"] *
    raw["spending_deviation_score"]
)

raw["velocity_geo"] = (
    raw["velocity_score"] *
    raw["geo_anomaly_score"]
)

raw["velocity_spending"] = (
    raw["velocity_score"] *
    raw["spending_deviation_score"]
)

raw["geo_spending"] = (
    raw["geo_anomaly_score"] *
    raw["spending_deviation_score"]
)

In [55]:
# ============================================================
# Missing Value Summary
# ============================================================

missing = (
    raw.isnull()
       .sum()
       .sort_values(ascending=False)
)

missing = missing[missing > 0]

display(missing)

fraud_type                     4820447
historical_std_amount          1773816
time_gap_minutes                896513
avg_txn_per_day                 896513
amount_median_ratio             896513
historical_median_amount        896513
historical_total_amount         896513
amount_pct_change               896513
amount_change                   896513
previous_amount                 896513
previous_timestamp              896513
amount_difference               896513
amount_ratio                    896513
historical_min_amount           896513
historical_max_amount           896513
historical_avg_amount           896513
time_gap_hours                  896513
time_since_last_transaction     896513
dtype: int64

In [56]:
history_features = [

    "historical_avg_amount",
    "historical_max_amount",
    "historical_min_amount",
    "historical_std_amount",
    "historical_median_amount",
    "previous_amount",
    "amount_change",
    "amount_pct_change",
    "amount_ratio",
    "amount_median_ratio",
    "historical_total_amount",
    "time_gap_minutes",
    "time_gap_hours",
    "avg_txn_per_day"

]

raw[history_features] = raw[history_features].fillna(0)

In [57]:
print(raw.shape)

(5000000, 52)


In [62]:
from pathlib import Path

OUTPUT = Path("../data")
OUTPUT.mkdir(exist_ok=True)

raw.to_csv(
    OUTPUT / "fraud_detection_enhanced.csv",
    index=False
)

print("Enhanced dataset saved.")
print(raw.shape)

Enhanced dataset saved.
(5000000, 52)


In [59]:
from pathlib import Path

OUTPUT = Path("../data")

print("Save folder:", OUTPUT.resolve())

Save folder: /Users/gyanvi/Downloads/fraud-detection-drift-aware/data


In [60]:
import os

print(os.listdir(OUTPUT))

['fraud_detection_enhanced.csv', 'feature_ranking.csv', 'processed', 'raw']


In [63]:
raw["log_amount"] = np.log1p(raw["amount"])

raw["hour"] = raw["timestamp"].dt.hour

raw["day"] = raw["timestamp"].dt.day

raw["day_of_week"] = raw["timestamp"].dt.dayofweek

raw["month"] = raw["timestamp"].dt.month

raw["is_weekend"] = raw["day_of_week"].isin([5, 6]).astype(int)

raw["is_night"] = raw["hour"].between(0, 5).astype(int)

raw["is_business_hours"] = raw["hour"].between(9, 17).astype(int)

raw["is_peak_hours"] = raw["hour"].between(17, 21).astype(int)

raw["is_late_night"] = raw["hour"].between(22, 23).astype(int)

raw["part_of_day"] = pd.cut(
    raw["hour"],
    bins=[-1, 5, 11, 17, 21, 24],
    labels=[
        "Night",
        "Morning",
        "Afternoon",
        "Evening",
        "Late Night"
    ]
)